# Gas Classification — Random Forest Baseline

Simple baseline: train a RandomForest on batches 1-9, validate with a batch-aware holdout (last batch out), then refit on all training data and predict on the hidden batch-10 test set.

In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score

train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
sample_sub = pd.read_csv("data/sample_submission.csv")

# "batch" is a nuisance/time variable, not a sensor reading, and test batch (10) is
# never seen in train — excluding it forces the model to rely on the actual sensor signal.
feature_cols = [c for c in train.columns if c not in ("measurement_id", "gas_class", "batch")]
train.shape, test.shape, sample_sub.shape

((10310, 132), (3600, 131), (3600, 2))

In [2]:
# Batch-aware validation: hold out the latest batch (9) to mimic predicting on a
# future, unseen batch (10) — a random shuffle split would leak drift information.
val_batch = train["batch"].max()
tr = train[train["batch"] != val_batch]
va = train[train["batch"] == val_batch]

X_tr, y_tr = tr[feature_cols], tr["gas_class"]
X_va, y_va = va[feature_cols], va["gas_class"]

clf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
clf.fit(X_tr, y_tr)

val_pred = clf.predict(X_va)
macro_f1 = f1_score(y_va, val_pred, average="macro")  # competition metric

print(f"Validation batch: {val_batch} ({len(va)} rows)")
print("Accuracy:", accuracy_score(y_va, val_pred))
print("Macro F1:", macro_f1)
print(classification_report(y_va, val_pred, digits=3))

Validation batch: 9 (470 rows)
Accuracy: 0.8957446808510638
Macro F1: 0.9005891286825172
              precision    recall  f1-score   support

           1      0.822     0.984     0.896        61
           2      1.000     1.000     1.000        55
           3      0.826     1.000     0.905       100
           4      0.886     0.933     0.909        75
           5      0.940     1.000     0.969        78
           6      0.983     0.574     0.725       101

    accuracy                          0.896       470
   macro avg      0.910     0.915     0.901       470
weighted avg      0.908     0.896     0.887       470



In [3]:
# Refit on all 9 labelled batches, then predict the hidden batch-10 test set.
final_clf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
final_clf.fit(train[feature_cols], train["gas_class"])

test_pred = final_clf.predict(test[feature_cols])

submission = pd.DataFrame({
    "measurement_id": test["measurement_id"],
    "gas_class": test_pred,
})
assert list(submission.columns) == list(sample_sub.columns)
assert len(submission) == len(sample_sub)

submission.to_csv("data/submission.csv", index=False)
submission.head()

,measurement_id,gas_class
0,G_B10_0001,6
1,G_B10_0002,2
2,G_B10_0003,4
3,G_B10_0004,2
4,G_B10_0005,4


## Drift & imbalance investigation

The baseline above validates at macro F1 ≈ 0.90 but is noticeably weak on class 6. Before building a more robust model, look at *why*: batch-to-batch feature scale drift, batch-conditional class scarcity, and a concentration shift between train and test.

In [4]:
import numpy as np

print("=== Feature scale drift across batches (raw magnitude, feat_1..feat_3) ===")
print(train.groupby("batch")[["feat_1", "feat_2", "feat_3"]].mean())

print("\n=== Class counts per batch (note: class 6 is ~absent from batches 3-5) ===")
print(pd.crosstab(train["batch"], train["gas_class"]))

print("\n=== Concentration by class (train) ===")
print(train.groupby("gas_class")["concentration"].agg(["mean", "min", "max"]))

print("\n=== Concentration: train vs test ===")
print(pd.DataFrame({
    "train": train["concentration"].describe(),
    "test": test["concentration"].describe(),
}))

=== Feature scale drift across batches (raw magnitude, feat_1..feat_3) ===
              feat_1     feat_2     feat_3
batch                                     
1      125598.154467   6.105952  27.857710
2      125670.729174   7.210686  31.498924
3       65254.447305  10.778013  17.058570
4       49495.964760  12.353687  15.044143
5       89478.024721  12.756489  24.550021
6       50208.586919   6.608333  13.508169
7       20516.308405   5.830753   5.365482
8        2067.326015   2.566515   0.482140
9       20687.490190   3.054345   5.270847

=== Class counts per batch (note: class 6 is ~absent from batches 3-5) ===
gas_class    1    2    3    4    5    6
batch                                  
1           90   98   83   30   70   74
2          164  334  100  109  532    5
3          365  490  216  240  275    0
4           64   43   12   30   12    0
5           28   40   20   46   63    0
6          514  574  110   29  606  467
7          649  662  360  744  630  568
8           30  

## Per-batch feature standardization

Raw sensor feature magnitudes drift by orders of magnitude across batches (e.g. `feat_1` mean swings from ~125,600 to ~2,000). Z-scoring each feature *within its own batch* removes that batch-specific scale/offset without touching labels, so it applies just as validly to the unlabeled test batch. Also add `log1p(concentration)` alongside raw `concentration` — concentration is right-skewed (1-1000) and test skews to higher values than train, so a log-scaled version may transfer better.

In [5]:
feat_cols = [c for c in train.columns if c.startswith("feat_")]


def batch_zscore(df, cols, batch_col="batch"):
    """Z-score each column within its own batch group (uses no labels)."""
    df = df.copy()
    grp = df.groupby(batch_col)[cols]
    mean = grp.transform("mean")
    std = grp.transform("std").replace(0, 1)  # avoid div-by-zero on constant columns
    df[cols] = (df[cols] - mean) / std
    return df


train_scaled = batch_zscore(train, feat_cols)
test_scaled = batch_zscore(test, feat_cols)

train_scaled["log1p_concentration"] = np.log1p(train_scaled["concentration"])
test_scaled["log1p_concentration"] = np.log1p(test_scaled["concentration"])

feature_cols_v2 = feat_cols + ["concentration", "log1p_concentration"]

# sanity check: per-batch means should now be ~0 for every batch
train_scaled.groupby("batch")[feat_cols[:3]].mean()

C:\Users\Pc\AppData\Local\Temp\ipykernel_34292\2270664802.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_scaled["log1p_concentration"] = np.log1p(train_scaled["concentration"])
C:\Users\Pc\AppData\Local\Temp\ipykernel_34292\2270664802.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_scaled["log1p_concentration"] = np.log1p(test_scaled["concentration"])


,feat_1,feat_2,feat_3
batch,,,
1,-3.118604e-18,8.283012e-17,1.372186e-17
2,-2.110673e-17,6.436884e-18,-7.759541e-18
3,8.449175e-17,3.192066e-17,2.877060e-17
4,-5.516636e-18,-1.379159e-17,-3.034150e-17
5,1.431455e-16,5.945611e-17,1.217300e-16
6,6.854420e-17,4.030592e-17,-2.896234e-18
7,6.787940e-17,-1.600958e-17,-3.997786e-17
8,1.888134e-18,4.531523e-18,-5.853217e-17
9,3.602319e-17,4.541285e-17,5.196789e-18


## Forward-chaining (walk-forward) cross-validation

Leave-one-batch-out is wrong for this problem: holding out batch 1 while training on batches 2-9 trains the model on *future* drift to predict the *past*, which never happens at deployment (the model only ever has access to batches before the one it's scoring). The correct scheme is walk-forward: for each batch, train only on the batches that came before it, then validate on it. The last fold (train on 1-8, validate on 9) is exactly the original single holdout; earlier folds add more (noisier, smaller-train-set) samples of "predict the next batch from only the past."

In [6]:
def forward_chaining_cv(df, feature_cols, model_factory, sample_weight_fn=None,
                         label_col="gas_class", batch_col="batch", label_offset=0):
    """Walk-forward validation: train on all batches before batch b, validate on b,
    for every batch after the first. Never trains on a batch that comes after the
    one being validated -- matches the real "predict the next unseen batch" task.

    sample_weight_fn receives the training-fold DATAFRAME (not just y), so weights
    can depend on batch age as well as class.
    label_offset: XGBoost requires 0-indexed class labels (ours run 1-6), so
    subtract it before fit and add it back after predict; 0 is a no-op for
    other estimators.
    """
    batches = sorted(df[batch_col].unique())
    scores = []
    for val_batch in batches[1:]:
        tr_fold = df[df[batch_col] < val_batch]
        va_fold = df[df[batch_col] == val_batch]
        X_tr, y_tr = tr_fold[feature_cols], tr_fold[label_col] - label_offset
        X_va, y_va = va_fold[feature_cols], va_fold[label_col]

        model = model_factory()
        fit_kwargs = {"sample_weight": sample_weight_fn(tr_fold)} if sample_weight_fn else {}
        model.fit(X_tr, y_tr, **fit_kwargs)

        preds = model.predict(X_va) + label_offset
        f1 = f1_score(y_va, preds, average="macro")
        scores.append((val_batch, len(va_fold), f1))
    return scores


def summarize_cv(scores, name):
    f1s = np.array([s[2] for s in scores])
    print(f"{name}: macro F1 = {f1s.mean():.4f} +/- {f1s.std():.4f}")
    for b, n, f1 in scores:
        print(f"  validate on batch {b} (n={n}): {f1:.4f}")
    return f1s.mean()

## Model comparison: scaling + class-balanced weighting + model family

Compare, under the same leave-one-batch-out CV:
- the original baseline config (raw features, no class weight) for reference,
- RandomForest with per-batch-scaled features + `class_weight="balanced"`,
- LightGBM with per-batch-scaled features + `class_weight="balanced"`,
- XGBoost with per-batch-scaled features + balanced `sample_weight` (XGBoost has no native `class_weight`).

Class-balanced weighting matters most for class 6, which is scarce in several batches.

In [7]:
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight

# name -> (train_df, test_df, feature_cols, model_factory, sample_weight_fn, label_offset)
configs = {
    "RF (baseline: raw, unweighted)": (
        train, test, feature_cols,
        lambda: RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
        None, 0,
    ),
    "RF (scaled + balanced)": (
        train_scaled, test_scaled, feature_cols_v2,
        lambda: RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1, class_weight="balanced"),
        None, 0,
    ),
    "LightGBM (scaled + balanced)": (
        train_scaled, test_scaled, feature_cols_v2,
        lambda: LGBMClassifier(n_estimators=300, random_state=42, class_weight="balanced", verbosity=-1),
        None, 0,
    ),
    "XGBoost (scaled + balanced)": (
        train_scaled, test_scaled, feature_cols_v2,
        lambda: XGBClassifier(n_estimators=300, random_state=42, eval_metric="mlogloss"),
        lambda df: compute_sample_weight("balanced", df["gas_class"]), 1,  # labels 1-6 -> 0-indexed for xgboost
    ),
}

results = {}
for name, (tr_df, _te_df, cols, factory, sw_fn, label_offset) in configs.items():
    scores = forward_chaining_cv(tr_df, cols, factory, sample_weight_fn=sw_fn, label_offset=label_offset)
    results[name] = summarize_cv(scores, name)

print("\n=== Comparison (mean macro F1 across the forward-chaining folds) ===")
for name, score in sorted(results.items(), key=lambda kv: -kv[1]):
    print(f"{name}: {score:.4f}")

best_name = max(results, key=results.get)
print("\nBest config:", best_name)

RF (baseline: raw, unweighted): macro F1 = 0.7800 +/- 0.1069
  validate on batch 2 (n=1244): 0.6784
  validate on batch 3 (n=1586): 0.9194
  validate on batch 4 (n=161): 0.7140
  validate on batch 5 (n=197): 0.9040
  validate on batch 6 (n=2300): 0.6244
  validate on batch 7 (n=3613): 0.7239
  validate on batch 8 (n=294): 0.7750
  validate on batch 9 (n=470): 0.9006
RF (scaled + balanced): macro F1 = 0.6923 +/- 0.1438
  validate on batch 2 (n=1244): 0.5559
  validate on batch 3 (n=1586): 0.6226
  validate on batch 4 (n=161): 0.7978
  validate on batch 5 (n=197): 0.9838
  validate on batch 6 (n=2300): 0.6121
  validate on batch 7 (n=3613): 0.7805
  validate on batch 8 (n=294): 0.5159
  validate on batch 9 (n=470): 0.6699
LightGBM (scaled + balanced): macro F1 = 0.6618 +/- 0.1964
  validate on batch 2 (n=1244): 0.3595
  validate on batch 3 (n=1586): 0.5699
  validate on batch 4 (n=161): 0.6940
  validate on batch 5 (n=197): 0.9800
  validate on batch 6 (n=2300): 0.6238
  validate on batc

## 2D drift-adaptive SVM ensemble (Vergara et al. 2012 style)

Two nested weighting levels, built on one-vs-one RBF-SVM:

1. **Within each batch's model** — a multiclass SVM is really k(k-1)/2 pairwise binary classifiers. Instead of combining them with equal weight, weight each pairwise classifier by its accuracy on a recent labelled reference set, then combine via **Wu-Lin-Weng pairwise coupling** (the same probability-estimation algorithm libsvm uses internally, generalized here to accept per-pair weights).
2. **Across batches** — keep one such multiclass model per training batch (not just the latest) and combine them into an ensemble, weighting each batch's model (`beta_t`) by its accuracy on the same reference set. Batches closer to "now" tend to score higher.

**Pre-aging**: weighting every model off only the single latest batch makes weights swing wildly (that batch's own model gets an inflated self-evaluation score). Method 2 from the paper pools *all* available batches as the reference set instead, diluting that bias. Both variants are implemented below for comparison.

Simplifications made here (noted rather than hidden): fixed SVC hyperparameters (`C=10, gamma="scale"`, RBF kernel) instead of the paper's grid-search + 10-fold CV — a full nested search across 8 chronological folds x 2 pre-aging settings x many (C, gamma) pairs would be very expensive for a first pass; and missing-class pairs (class 6 is absent from batches 3-5) get a neutral 0.5 weight rather than being dropped. Features are scaled to [-1, 1] with a single scaler fit on all *available past* batches (not per-batch — per-batch scaling was already shown above to destroy real dose-response signal for tree models, and RBF-SVM needs *some* global scaling to avoid `feat_1`'s huge raw magnitude dominating the kernel distance).

In [8]:
import warnings

from sklearn.svm import SVC
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings("ignore", category=FutureWarning)  # SVC(probability=True) deprecation noise


def wu_lin_weng_couple(R, W, max_iter=100, tol=1e-6):
    """Weighted pairwise coupling (Wu, Lin & Weng 2004, Algorithm 2), generalized
    with a per-pair weight W_st on each term of the least-squares objective:
        min_p  sum_{s<t} W_st * (r_ts*p_s - r_st*p_t)^2 ,  s.t. sum(p)=1, p>=0

    R: (n_samples, k, k) pairwise probabilities, R[:, i, j] = P(class i | class i or j)
    W: (k, k) symmetric static weights per pair (diagonal unused)
    Returns (n_samples, k) combined class probabilities.
    """
    n, k, _ = R.shape
    Q = np.zeros((n, k, k))
    for t in range(k):
        Q[:, t, t] = sum(W[s, t] * R[:, s, t] ** 2 for s in range(k) if s != t)
        for s in range(k):
            if s != t:
                Q[:, t, s] = -W[t, s] * R[:, t, s] * R[:, s, t]

    p = np.full((n, k), 1.0 / k)
    for _ in range(max_iter):
        Qp = np.einsum("nij,nj->ni", Q, p)
        pQp = np.einsum("ni,ni->n", p, Qp)
        p_new = np.empty_like(p)
        for t in range(k):
            sum_other = Qp[:, t] - Q[:, t, t] * p[:, t]
            p_new[:, t] = (-sum_other + pQp) / np.clip(Q[:, t, t], 1e-12, None)
        p_new = np.clip(p_new, 0, None)
        row_sums = p_new.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1.0
        p_new /= row_sums
        if np.abs(p_new - p).max() < tol:
            p = p_new
            break
        p = p_new
    return p

In [9]:
class DriftAdaptiveSVMEnsemble:
    """One one-vs-one RBF-SVM per training batch, combined via weighted pairwise
    coupling within each batch and weighted averaging across batches (Vergara et al. 2012).
    """

    def __init__(self, classes, svc_kwargs=None, pre_aging=True):
        self.classes = list(classes)
        self.svc_kwargs = svc_kwargs or dict(kernel="rbf", C=10, gamma="scale", probability=True, random_state=42)
        self.pre_aging = pre_aging
        self.batch_models_ = {}
        self.batch_weights_ = {}  # batch -> (W matrix, beta_t)

    def _fit_ovo(self, X, y):
        clfs = {}
        present = sorted(set(y))
        for a in range(len(present)):
            for b in range(a + 1, len(present)):
                ci, cj = present[a], present[b]
                mask = np.isin(y, [ci, cj])
                clfs[(ci, cj)] = SVC(**self.svc_kwargs).fit(X[mask], y[mask])
        return clfs

    def _pairwise_r(self, pairwise_clfs, X):
        k = len(self.classes)
        idx = {c: i for i, c in enumerate(self.classes)}
        R = np.zeros((X.shape[0], k, k))
        for (ci, cj), clf in pairwise_clfs.items():
            proba = clf.predict_proba(X)
            pi = proba[:, list(clf.classes_).index(ci)]
            R[:, idx[ci], idx[cj]] = pi
            R[:, idx[cj], idx[ci]] = 1 - pi
        return R

    def _pairwise_weights(self, pairwise_clfs, X_ref, y_ref):
        k = len(self.classes)
        idx = {c: i for i, c in enumerate(self.classes)}
        W = np.zeros((k, k))
        for (ci, cj), clf in pairwise_clfs.items():
            mask = np.isin(y_ref, [ci, cj])
            w = accuracy_score(y_ref[mask], clf.predict(X_ref[mask])) if mask.sum() else 0.5
            W[idx[ci], idx[cj]] = w
            W[idx[cj], idx[ci]] = w
        return W

    def fit(self, df, feature_cols, label_col="gas_class", batch_col="batch"):
        batches = sorted(df[batch_col].unique())
        for b in batches:
            sub = df[df[batch_col] == b]
            self.batch_models_[b] = self._fit_ovo(sub[feature_cols].values, sub[label_col].values)

        ref_df = df if self.pre_aging else df[df[batch_col] == batches[-1]]
        X_ref, y_ref = ref_df[feature_cols].values, ref_df[label_col].values

        for b in batches:
            pairwise_clfs = self.batch_models_[b]
            W = self._pairwise_weights(pairwise_clfs, X_ref, y_ref)
            R_ref = self._pairwise_r(pairwise_clfs, X_ref)
            P_ref = wu_lin_weng_couple(R_ref, W)
            beta = accuracy_score(y_ref, np.array(self.classes)[P_ref.argmax(axis=1)])
            self.batch_weights_[b] = (W, beta)
        return self

    def predict_proba(self, X):
        k = len(self.classes)
        combined = np.zeros((X.shape[0], k))
        beta_sum = 0.0
        for b, pairwise_clfs in self.batch_models_.items():
            W, beta = self.batch_weights_[b]
            R = self._pairwise_r(pairwise_clfs, X)
            combined += beta * wu_lin_weng_couple(R, W)
            beta_sum += beta
        return combined / (beta_sum if beta_sum else 1.0)

    def predict(self, X):
        return np.array(self.classes)[self.predict_proba(X).argmax(axis=1)]

In [10]:
def forward_chaining_cv_svm_ensemble(df, feature_cols, classes, pre_aging, batch_col="batch", label_col="gas_class"):
    """Same walk-forward scheme as forward_chaining_cv, but the SVM ensemble needs
    per-batch grouping at fit time (not just a merged X/y), so it gets its own harness.
    Feature scaling is refit each fold on only the batches available at that point
    (no leakage from the validation batch or the future).
    """
    batches = sorted(df[batch_col].unique())
    scores = []
    for val_batch in batches[1:]:
        tr_fold = df[df[batch_col] < val_batch]
        va_fold = df[df[batch_col] == val_batch]

        scaler = MinMaxScaler(feature_range=(-1, 1))
        tr_scaled = tr_fold.copy()
        tr_scaled[feature_cols] = scaler.fit_transform(tr_fold[feature_cols])
        X_va_scaled = scaler.transform(va_fold[feature_cols])

        ens = DriftAdaptiveSVMEnsemble(classes=classes, pre_aging=pre_aging)
        ens.fit(tr_scaled, feature_cols, label_col=label_col, batch_col=batch_col)
        preds = ens.predict(X_va_scaled)

        f1 = f1_score(va_fold[label_col], preds, average="macro")
        scores.append((val_batch, len(va_fold), f1))
    return scores


classes = sorted(train["gas_class"].unique())

for pre_aging, name in [(True, "SVM 2D ensemble (pre-aging)"), (False, "SVM 2D ensemble (no pre-aging)")]:
    scores = forward_chaining_cv_svm_ensemble(train, feature_cols, classes, pre_aging=pre_aging)
    results[name] = summarize_cv(scores, name)

print("\n=== Updated comparison (mean macro F1 across the forward-chaining folds) ===")
for name, score in sorted(results.items(), key=lambda kv: -kv[1]):
    print(f"{name}: {score:.4f}")

best_name = max(results, key=results.get)
print("\nBest config overall:", best_name)

SVM 2D ensemble (pre-aging): macro F1 = 0.7575 +/- 0.1994
  validate on batch 2 (n=1244): 0.5220
  validate on batch 3 (n=1586): 0.8808
  validate on batch 4 (n=161): 0.8497
  validate on batch 5 (n=197): 0.7247
  validate on batch 6 (n=2300): 0.3600
  validate on batch 7 (n=3613): 0.8179
  validate on batch 8 (n=294): 0.9533
  validate on batch 9 (n=470): 0.9519
SVM 2D ensemble (no pre-aging): macro F1 = 0.7559 +/- 0.1832
  validate on batch 2 (n=1244): 0.5220
  validate on batch 3 (n=1586): 0.8619
  validate on batch 4 (n=161): 0.8537
  validate on batch 5 (n=197): 0.7253
  validate on batch 6 (n=2300): 0.4086
  validate on batch 7 (n=3613): 0.7901
  validate on batch 8 (n=294): 0.9322
  validate on batch 9 (n=470): 0.9536

=== Updated comparison (mean macro F1 across the forward-chaining folds) ===
RF (baseline: raw, unweighted): 0.7800
SVM 2D ensemble (pre-aging): 0.7575
SVM 2D ensemble (no pre-aging): 0.7559
RF (scaled + balanced): 0.6923
XGBoost (scaled + balanced): 0.6771
LightG

## Four more drift-compensation families

All evaluated under the **same** forward-chaining CV, so every number below is directly comparable to the RF baseline's 0.780.

| Family | Technique | What it does here |
|---|---|---|
| Signal preprocessing | **Baseline correction** | Divide each sensor's 8 descriptors by that sensor's own steady-state ΔR, cancelling multiplicative sensor gain |
| Component analysis | **OSC / component correction** | Find the feature-space directions along which the *batch means* drift, and project them out |
| Domain adaptation | **CORAL** and **DANN** | Align source (past batches) to target (the batch being predicted) — closed-form covariance alignment, and a domain-adversarial neural net |
| Ensemble learning | **Dynamic classifier ensembles** | Per-batch models soft-voted by recency × accuracy, and a single RF with recency-decayed sample weights |

Two rules kept throughout: nothing trains on a batch later than the one it predicts, and adaptation may use the target batch's **features** but never its labels.

In [11]:
# --- 1. Baseline correction (signal preprocessing) ---------------------------
# Layout is sensor-major: 16 sensors x 8 descriptors. feat_1, feat_9, ..., feat_121
# are the steady-state dR columns (mean |value| > 1000; the rest are small EMA
# transients). Dividing each sensor's descriptors by its own steady-state value is
# the feature-space analogue of "divide the peak response by the clean-air baseline":
# it cancels per-sensor multiplicative gain, which is exactly what ages over time.
RF = lambda: RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)


def baseline_correct(df):
    out = df[[c for c in ("measurement_id", "batch", "concentration", "gas_class") if c in df.columns]].copy()
    new = {}
    for s in range(16):
        block = [f"feat_{s * 8 + j + 1}" for j in range(8)]
        base = df[block[0]].values
        safe = np.where(np.abs(base) < 1e-9, 1e-9, base)
        for j in range(1, 8):  # j=0 would be base/base = all ones, so skip it
            new[f"s{s}_r{j}"] = df[block[j]].values / safe
        new[f"s{s}_logbase"] = np.log1p(np.abs(base))  # keep absolute scale, log-compressed
    return pd.concat([out, pd.DataFrame(new, index=df.index)], axis=1)


train_bc, test_bc = baseline_correct(train), baseline_correct(test)
feature_cols_bc = [c for c in train_bc.columns if c.startswith("s")] + ["concentration"]

results["Baseline correction (per-sensor ratios)"] = summarize_cv(
    forward_chaining_cv(train_bc, feature_cols_bc, RF), "Baseline correction (per-sensor ratios)")

Baseline correction (per-sensor ratios): macro F1 = 0.8265 +/- 0.1445
  validate on batch 2 (n=1244): 0.5946
  validate on batch 3 (n=1586): 0.9547
  validate on batch 4 (n=161): 0.8295
  validate on batch 5 (n=197): 0.9691
  validate on batch 6 (n=2300): 0.6256
  validate on batch 7 (n=3613): 0.7484
  validate on batch 8 (n=294): 0.9145
  validate on batch 9 (n=470): 0.9757


In [12]:
# --- 2. OSC / component correction -------------------------------------------
from sklearn.preprocessing import StandardScaler
from scipy import linalg


def forward_chaining_cv_transductive(df, feature_cols, predict_fn, label_col="gas_class", batch_col="batch"):
    """Walk-forward CV for methods that need the target batch's features at fit time.
    predict_fn(tr_fold_df, X_va, feature_cols) -> preds. Target FEATURES may be used
    (legitimate transductive adaptation); target labels never are.
    """
    batches = sorted(df[batch_col].unique())
    scores = []
    for val_batch in batches[1:]:
        tr_fold = df[df[batch_col] < val_batch]
        va_fold = df[df[batch_col] == val_batch]
        preds = predict_fn(tr_fold, va_fold[feature_cols].values, feature_cols)
        scores.append((val_batch, len(va_fold), f1_score(va_fold[label_col], preds, average="macro")))
    return scores


def osc_projector(tr_fold, X_tr_scaled, k, batch_col="batch"):
    """Drift subspace = principal directions of variation ACROSS per-batch mean vectors.
    Returns a projector that removes the top-k of them."""
    cent = np.vstack([X_tr_scaled[tr_fold[batch_col].values == b].mean(axis=0)
                      for b in sorted(tr_fold[batch_col].unique())])
    cent = cent - cent.mean(axis=0)
    if cent.shape[0] <= k:
        return np.eye(X_tr_scaled.shape[1])
    _, _, Vt = np.linalg.svd(cent, full_matrices=False)
    D = Vt[:k]
    return np.eye(X_tr_scaled.shape[1]) - D.T @ D


def make_osc_predict_fn(k):
    def predict_fn(tr_fold, X_va, cols):
        sc = StandardScaler().fit(tr_fold[cols])
        X_tr, X_va_s = sc.transform(tr_fold[cols]), sc.transform(X_va)
        P = osc_projector(tr_fold, X_tr, k)
        return RF().fit(X_tr @ P, tr_fold["gas_class"]).predict(X_va_s @ P)
    return predict_fn


for k in (1, 2, 3):
    name = f"OSC / component correction (k={k})"
    results[name] = summarize_cv(
        forward_chaining_cv_transductive(train, feature_cols, make_osc_predict_fn(k)), name)

C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler wa

OSC / component correction (k=1): macro F1 = 0.8292 +/- 0.1218
  validate on batch 2 (n=1244): 0.6784
  validate on batch 3 (n=1586): 0.9659
  validate on batch 4 (n=161): 0.7929
  validate on batch 5 (n=197): 0.9895
  validate on batch 6 (n=2300): 0.6349
  validate on batch 7 (n=3613): 0.7812
  validate on batch 8 (n=294): 0.8689
  validate on batch 9 (n=470): 0.9221


C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler wa

OSC / component correction (k=2): macro F1 = 0.8467 +/- 0.1243
  validate on batch 2 (n=1244): 0.6784
  validate on batch 3 (n=1586): 0.9194
  validate on batch 4 (n=161): 0.7813
  validate on batch 5 (n=197): 0.9933
  validate on batch 6 (n=2300): 0.6413
  validate on batch 7 (n=3613): 0.8574
  validate on batch 8 (n=294): 0.9542
  validate on batch 9 (n=470): 0.9484


C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler wa

OSC / component correction (k=3): macro F1 = 0.8454 +/- 0.1334
  validate on batch 2 (n=1244): 0.6784
  validate on batch 3 (n=1586): 0.9194
  validate on batch 4 (n=161): 0.7140
  validate on batch 5 (n=197): 0.9838
  validate on batch 6 (n=2300): 0.6418
  validate on batch 7 (n=3613): 0.8980
  validate on batch 8 (n=294): 0.9564
  validate on batch 9 (n=470): 0.9717


In [13]:
# --- 3a. CORAL (closed-form domain adaptation) --------------------------------
# Whiten the source covariance, recolour it with the target's: Xs @ Cs^-1/2 @ Ct^1/2.
# Uses target FEATURES only. Two variants plus an ablation, because the naive version
# fails badly and it is worth showing why rather than just reporting a bad number.
def coral_shared_scaler(tr_fold, X_va, cols, eps=1.0):
    """Naive: one scaler fit on the source, so the drifted target keeps a shifted mean.
    CORAL aligns only 2nd-order statistics, so that leftover mean gap is never fixed."""
    sc = StandardScaler().fit(tr_fold[cols])
    Xs, Xt = sc.transform(tr_fold[cols]), sc.transform(X_va)
    d = Xs.shape[1]
    A = np.real(linalg.sqrtm(linalg.inv(np.cov(Xs, rowvar=False) + eps * np.eye(d)))
                @ linalg.sqrtm(np.cov(Xt, rowvar=False) + eps * np.eye(d)))
    return RF().fit(Xs @ A, tr_fold["gas_class"]).predict(Xt)


def coral_per_domain(tr_fold, X_va, cols, eps=1e-3):
    """Centre/scale each domain by its OWN statistics first, then covariance-align."""
    Xs, Xt = StandardScaler().fit_transform(tr_fold[cols]), StandardScaler().fit_transform(X_va)
    d = Xs.shape[1]
    A = np.real(linalg.sqrtm(linalg.inv(np.cov(Xs, rowvar=False) + eps * np.eye(d)))
                @ linalg.sqrtm(np.cov(Xt, rowvar=False) + eps * np.eye(d)))
    return RF().fit(Xs @ A, tr_fold["gas_class"]).predict(Xt)


def per_domain_standardize_only(tr_fold, X_va, cols):
    """Ablation: per-domain standardisation with NO covariance alignment. If this ties
    coral_per_domain, then the alignment step itself is contributing nothing."""
    Xs, Xt = StandardScaler().fit_transform(tr_fold[cols]), StandardScaler().fit_transform(X_va)
    return RF().fit(Xs, tr_fold["gas_class"]).predict(Xt)


for fn, name in [(coral_shared_scaler, "CORAL (naive: shared source scaler)"),
                 (coral_per_domain, "CORAL (per-domain centering)"),
                 (per_domain_standardize_only, "Ablation: per-domain standardization only")]:
    results[name] = summarize_cv(forward_chaining_cv_transductive(train, feature_cols, fn), name)

C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler wa

CORAL (naive: shared source scaler): macro F1 = 0.4964 +/- 0.2135
  validate on batch 2 (n=1244): 0.4694
  validate on batch 3 (n=1586): 0.5656
  validate on batch 4 (n=161): 0.7244
  validate on batch 5 (n=197): 0.7646
  validate on batch 6 (n=2300): 0.4460
  validate on batch 7 (n=3613): 0.6217
  validate on batch 8 (n=294): 0.3096
  validate on batch 9 (n=470): 0.0696
CORAL (per-domain centering): macro F1 = 0.6292 +/- 0.1898
  validate on batch 2 (n=1244): 0.4182
  validate on batch 3 (n=1586): 0.6072
  validate on batch 4 (n=161): 0.7933
  validate on batch 5 (n=197): 0.9729
  validate on batch 6 (n=2300): 0.5737
  validate on batch 7 (n=3613): 0.7853
  validate on batch 8 (n=294): 0.4516
  validate on batch 9 (n=470): 0.4316
Ablation: per-domain standardization only: macro F1 = 0.6299 +/- 0.1444
  validate on batch 2 (n=1244): 0.5467
  validate on batch 3 (n=1586): 0.6241
  validate on batch 4 (n=161): 0.5807
  validate on batch 5 (n=197): 0.9794
  validate on batch 6 (n=2300): 0

In [14]:
# --- 3b. DANN (domain-adversarial neural network) -----------------------------
# Shared trunk + label head + domain head behind a gradient-reversal layer. The trunk
# is pushed to produce features the domain discriminator CANNOT separate, i.e. features
# that look the same for old batches and the batch being predicted.
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Neural models on: {DEVICE}"
      + (f"  ({torch.cuda.get_device_name(0)}, sm_{'%d%d' % torch.cuda.get_device_capability(0)})"
         if torch.cuda.is_available() else "  [WARNING: no GPU -- this will be very slow]"))


class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambd):
        ctx.lambd = lambd
        return x.view_as(x)

    @staticmethod
    def backward(ctx, g):
        return -ctx.lambd * g, None  # the whole trick: flip the gradient sign


class DANN(nn.Module):
    def __init__(self, d_in, n_classes=6):
        super().__init__()
        self.trunk = nn.Sequential(nn.Linear(d_in, 128), nn.ReLU(), nn.Dropout(0.3),
                                   nn.Linear(128, 64), nn.ReLU())
        self.label_head = nn.Linear(64, n_classes)
        self.domain_head = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 2))

    def forward(self, x, lambd=0.0):
        f = self.trunk(x)
        return self.label_head(f), self.domain_head(GradReverse.apply(f, lambd))


def fit_dann(Xs, ys, Xt, epochs=40, bs=1024, lr=1e-3, adversarial=True):
    model = DANN(Xs.shape[1]).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    ce = nn.CrossEntropyLoss()
    Xs_t = torch.tensor(Xs, dtype=torch.float32, device=DEVICE)
    ys_t = torch.tensor(ys, dtype=torch.long, device=DEVICE)
    Xt_t = torch.tensor(Xt, dtype=torch.float32, device=DEVICE)

    for ep in range(epochs):
        p = ep / max(epochs - 1, 1)
        lambd = (2.0 / (1.0 + np.exp(-10 * p)) - 1.0) if adversarial else 0.0  # standard ramp-up
        model.train()
        for idx in torch.randperm(len(Xs_t), device=DEVICE).split(bs):
            xb, yb = Xs_t[idx], ys_t[idx]
            logits_s, dom_s = model(xb, lambd)
            loss = ce(logits_s, yb)
            if adversarial:
                tb = Xt_t[torch.randint(0, len(Xt_t), (len(idx),), device=DEVICE)]
                _, dom_t = model(tb, lambd)
                dom_y = torch.cat([torch.zeros(len(xb), dtype=torch.long, device=DEVICE),
                                   torch.ones(len(tb), dtype=torch.long, device=DEVICE)])
                loss = loss + ce(torch.cat([dom_s, dom_t]), dom_y)
            opt.zero_grad()
            loss.backward()
            opt.step()

    model.eval()
    with torch.no_grad():
        _, ds = model(Xs_t, 0.0)
        _, dt = model(Xt_t, 0.0)
        dom_acc = ((ds.argmax(1) == 0).sum() + (dt.argmax(1) == 1).sum()).item() / (len(Xs_t) + len(Xt_t))
        src_acc = (model(Xs_t, 0.0)[0].argmax(1) == ys_t).float().mean().item()
        preds = model(Xt_t, 0.0)[0].argmax(1).cpu().numpy()
    return preds, dom_acc, src_acc


def run_dann(adversarial, name):
    """Reports two sanity numbers per fold: domain_acc (0.5 => discriminator fooled,
    i.e. adaptation really happened) and src_acc (must beat 1/6 or the net didn't learn)."""
    batches = sorted(train["batch"].unique())
    scores, doms, srcs = [], [], []
    for val_batch in batches[1:]:
        tr_fold, va_fold = train[train["batch"] < val_batch], train[train["batch"] == val_batch]
        sc = StandardScaler().fit(tr_fold[feature_cols])
        preds, dom_acc, src_acc = fit_dann(
            sc.transform(tr_fold[feature_cols]), tr_fold["gas_class"].values - 1,
            sc.transform(va_fold[feature_cols]), adversarial=adversarial)
        scores.append((val_batch, len(va_fold), f1_score(va_fold["gas_class"], preds + 1, average="macro")))
        doms.append(dom_acc)
        srcs.append(src_acc)
    mean = summarize_cv(scores, name)
    print(f"  sanity: mean domain_acc={np.mean(doms):.3f} (0.5 = fully fooled), "
          f"mean source_acc={np.mean(srcs):.3f} (0.167 = random)")
    return mean


results["DANN (domain-adversarial)"] = run_dann(True, "DANN (domain-adversarial)")
results["MLP control (no adaptation)"] = run_dann(False, "MLP control (no adaptation)")

Neural models on: cuda  (NVIDIA GeForce RTX 5060 Ti, sm_120)
DANN (domain-adversarial): macro F1 = 0.7286 +/- 0.1293
  validate on batch 2 (n=1244): 0.6209
  validate on batch 3 (n=1586): 0.5882
  validate on batch 4 (n=161): 0.6812
  validate on batch 5 (n=197): 0.9933
  validate on batch 6 (n=2300): 0.6024
  validate on batch 7 (n=3613): 0.7493
  validate on batch 8 (n=294): 0.8441
  validate on batch 9 (n=470): 0.7495
  sanity: mean domain_acc=0.544 (0.5 = fully fooled), mean source_acc=0.975 (0.167 = random)
MLP control (no adaptation): macro F1 = 0.7872 +/- 0.1052
  validate on batch 2 (n=1244): 0.6838
  validate on batch 3 (n=1586): 0.7493
  validate on batch 4 (n=161): 0.7759
  validate on batch 5 (n=197): 0.9826
  validate on batch 6 (n=2300): 0.6521
  validate on batch 7 (n=3613): 0.7805
  validate on batch 8 (n=294): 0.9230
  validate on batch 9 (n=470): 0.7507
  sanity: mean domain_acc=0.363 (0.5 = fully fooled), mean source_acc=0.985 (0.167 = random)


In [15]:
# --- 4. Dynamic classifier ensembles ------------------------------------------
class DynamicRFEnsemble:
    """One RF per batch, soft-voted by (recency decay x accuracy on the most recent
    labelled batch). Same interface as DriftAdaptiveSVMEnsemble above.

    Expect this to inherit the SVM ensemble's batch-6 weakness for the same structural
    reason: batches that contain no class-6 rows contribute no class-6 signal at all.
    """

    def __init__(self, classes, decay=0.3, n_estimators=200):
        self.classes = list(classes)
        self.decay = decay
        self.n_estimators = n_estimators

    def fit(self, df, feature_cols, label_col="gas_class", batch_col="batch"):
        batches = sorted(df[batch_col].unique())
        latest = df[df[batch_col] == batches[-1]]
        self.models_, self.weights_ = {}, {}
        for b in batches:
            sub = df[df[batch_col] == b]
            m = RandomForestClassifier(n_estimators=self.n_estimators, random_state=42, n_jobs=-1)
            m.fit(sub[feature_cols], sub[label_col])
            acc = accuracy_score(latest[label_col], m.predict(latest[feature_cols]))
            self.models_[b] = m
            self.weights_[b] = np.exp(-self.decay * (batches[-1] - b)) * acc
        return self

    def predict(self, X):
        P = np.zeros((len(X), len(self.classes)))
        for b, m in self.models_.items():
            proba = m.predict_proba(X)
            p = np.zeros((len(X), len(self.classes)))
            for i, c in enumerate(m.classes_):  # a batch may not contain every class
                p[:, self.classes.index(c)] = proba[:, i]
            P += self.weights_[b] * p
        return np.array(self.classes)[P.argmax(axis=1)]


results["Dynamic RF ensemble (per-batch, recency x acc)"] = summarize_cv(
    forward_chaining_cv_transductive(
        train, feature_cols,
        lambda tr, X_va, cols: DynamicRFEnsemble(classes).fit(tr, cols).predict(X_va)),
    "Dynamic RF ensemble (per-batch, recency x acc)")

# Same "favour recent data" idea without partitioning: one RF, exponentially
# recency-decayed sample weights. Never starves a class that is missing from some batch.
def make_recency_weights(half_life):
    return lambda tr_fold: 0.5 ** ((tr_fold["batch"].max() - tr_fold["batch"].values) / half_life)


for hl in (1.0, 2.0, 4.0):
    name = f"Recency-weighted RF (half-life={hl} batches)"
    results[name] = summarize_cv(
        forward_chaining_cv(train, feature_cols, RF, sample_weight_fn=make_recency_weights(hl)), name)

C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feat

Dynamic RF ensemble (per-batch, recency x acc): macro F1 = 0.7744 +/- 0.1639
  validate on batch 2 (n=1244): 0.6461
  validate on batch 3 (n=1586): 0.9110
  validate on batch 4 (n=161): 0.8006
  validate on batch 5 (n=197): 0.9285
  validate on batch 6 (n=2300): 0.4533
  validate on batch 7 (n=3613): 0.6557
  validate on batch 8 (n=294): 0.8425
  validate on batch 9 (n=470): 0.9577
Recency-weighted RF (half-life=1.0 batches): macro F1 = 0.7630 +/- 0.1166
  validate on batch 2 (n=1244): 0.6505
  validate on batch 3 (n=1586): 0.9211
  validate on batch 4 (n=161): 0.7775
  validate on batch 5 (n=197): 0.9352
  validate on batch 6 (n=2300): 0.6122
  validate on batch 7 (n=3613): 0.6317
  validate on batch 8 (n=294): 0.7856
  validate on batch 9 (n=470): 0.7902
Recency-weighted RF (half-life=2.0 batches): macro F1 = 0.7802 +/- 0.1098
  validate on batch 2 (n=1244): 0.6505
  validate on batch 3 (n=1586): 0.9154
  validate on batch 4 (n=161): 0.7843
  validate on batch 5 (n=197): 0.9257
  val

In [16]:
# Refit recipes for the methods that are not simple (fit(X, y) -> predict) configs.
# Each takes the full labelled train set + the test set and returns batch-10 predictions.
def refit_osc(k):
    def fit_predict(train_df, test_df):
        sc = StandardScaler().fit(train_df[feature_cols])
        X_tr, X_te = sc.transform(train_df[feature_cols]), sc.transform(test_df[feature_cols])
        P = osc_projector(train_df, X_tr, k)  # drift directions from the 9 train batches
        return RF().fit(X_tr @ P, train_df["gas_class"]).predict(X_te @ P)
    return fit_predict


final_fitters = {
    "Baseline correction (per-sensor ratios)":
        lambda tr, te: RF().fit(baseline_correct(tr)[feature_cols_bc], tr["gas_class"])
                          .predict(baseline_correct(te)[feature_cols_bc]),
    "CORAL (naive: shared source scaler)":
        lambda tr, te: coral_shared_scaler(tr, te[feature_cols].values, feature_cols),
    "CORAL (per-domain centering)":
        lambda tr, te: coral_per_domain(tr, te[feature_cols].values, feature_cols),
    "Ablation: per-domain standardization only":
        lambda tr, te: per_domain_standardize_only(tr, te[feature_cols].values, feature_cols),
    "Dynamic RF ensemble (per-batch, recency x acc)":
        lambda tr, te: DynamicRFEnsemble(classes).fit(tr, feature_cols).predict(te[feature_cols].values),
    **{f"OSC / component correction (k={k})": refit_osc(k) for k in (1, 2, 3)},
    **{f"Recency-weighted RF (half-life={hl} batches)":
       (lambda hl: lambda tr, te: RF().fit(tr[feature_cols], tr["gas_class"],
                                           sample_weight=make_recency_weights(hl)(tr))
                                      .predict(te[feature_cols]))(hl)
       for hl in (1.0, 2.0, 4.0)},
}

print("=== FULL COMPARISON: mean macro F1 over the 8 forward-chaining folds ===")
for name, score in sorted(results.items(), key=lambda kv: -kv[1]):
    flag = " <-- beats RF baseline" if score > results["RF (baseline: raw, unweighted)"] else ""
    print(f"  {score:.4f}  {name}{flag}")

best_name = max(results, key=results.get)
print(f"\nBest overall: {best_name} ({results[best_name]:.4f})")

=== FULL COMPARISON: mean macro F1 over the 8 forward-chaining folds ===
  0.8467  OSC / component correction (k=2) <-- beats RF baseline
  0.8454  OSC / component correction (k=3) <-- beats RF baseline
  0.8292  OSC / component correction (k=1) <-- beats RF baseline
  0.8265  Baseline correction (per-sensor ratios) <-- beats RF baseline
  0.7872  MLP control (no adaptation) <-- beats RF baseline
  0.7802  Recency-weighted RF (half-life=2.0 batches) <-- beats RF baseline
  0.7800  RF (baseline: raw, unweighted)
  0.7771  Recency-weighted RF (half-life=4.0 batches)
  0.7744  Dynamic RF ensemble (per-batch, recency x acc)
  0.7630  Recency-weighted RF (half-life=1.0 batches)
  0.7575  SVM 2D ensemble (pre-aging)
  0.7559  SVM 2D ensemble (no pre-aging)
  0.7286  DANN (domain-adversarial)
  0.6923  RF (scaled + balanced)
  0.6771  XGBoost (scaled + balanced)
  0.6618  LightGBM (scaled + balanced)
  0.6299  Ablation: per-domain standardization only
  0.6292  CORAL (per-domain centering)
  

## Variational Beam Search — Bayesian online learning with change detection

Standard Bayesian online learning (last posterior becomes the next prior) becomes overconfident under drift and stops adapting — "catastrophic remembering". VBS fixes that with an explicit binary **change variable** `s_t` per batch:

- `s_t = 0` → prior is the previous posterior unchanged.
- `s_t = 1` → prior is the previous posterior **tempered**: precision scaled by `β < 1`, i.e. covariance broadened to `Σ/β`, which softens confidence and lets the new batch dominate.
- `p(s_t = 1) = sigmoid(log-evidence ratio + ξ₀)`, comparing how well the new batch is explained under the changed vs unchanged prior, with `ξ₀` the prior log-odds of a change.

**Greedy (K=1)** commits to the more probable `s_t` immediately. **Beam search (K=3)** keeps the top-K *decision sequences* by cumulative log-evidence, so an early wrong guess about a change point can be corrected in hindsight, with a diversification step to stop the beam collapsing onto near-identical hypotheses.

Model: Bayesian linear regression onto one-hot targets, predicting by argmax. The 6 outputs share one precision matrix, so the closed-form natural-parameter update is just `Λ += XᵀX/σ²`, `B += XᵀY/σ²` — and the marginal likelihood needed for the change variable is exact.

**This is a linear model, so expect it to trail RF/OSC on absolute macro-F1.** The result that actually tests VBS is whether it beats its *own* baselines — "train once on batch 1, never update" and "full retrain every batch" — using the identical regression.

In [17]:
# --- 5. VBS: Bayesian linear regression with change detection -----------------
VBS_CLASSES = list(classes)
NK = len(VBS_CLASSES)


def _onehot(y):
    Y = np.zeros((len(y), NK))
    for i, c in enumerate(VBS_CLASSES):
        Y[y == c, i] = 1.0
    return Y


def _design(X):
    return np.hstack([X, np.ones((len(X), 1))])  # bias column


def bayes_log_evidence(Lam0, B0, X, Y, noise=1.0):
    """Exact Gaussian-linear marginal likelihood in natural-parameter form, summed
    over the NK outputs that share Lam0. This is what the change variable compares."""
    LamN = Lam0 + (X.T @ X) / noise
    BN = B0 + (X.T @ Y) / noise
    s0, ld0 = np.linalg.slogdet(Lam0)
    sN, ldN = np.linalg.slogdet(LamN)
    if s0 <= 0 or sN <= 0:
        return -np.inf
    q0 = np.sum(B0 * np.linalg.solve(Lam0, B0))
    qN = np.sum(BN * np.linalg.solve(LamN, BN))
    return (0.5 * NK * (ld0 - ldN) + 0.5 * (qN - q0)
            - 0.5 * np.sum(Y ** 2) / noise - 0.5 * len(X) * NK * np.log(2 * np.pi * noise))


def _posterior(Lam0, B0, X, Y, noise=1.0):
    return Lam0 + (X.T @ X) / noise, B0 + (X.T @ Y) / noise


class VBSModel:
    """Fitted VBS state; .predict() matches the sklearn-ish interface used elsewhere."""

    def __init__(self, sc, Lam, B, flags, evidence):
        self.sc, self.Lam, self.B = sc, Lam, B
        self.change_flags, self.cum_evidence = flags, evidence

    def predict(self, X):
        Xd = _design(self.sc.transform(X))
        return np.array(VBS_CLASSES)[(Xd @ np.linalg.solve(self.Lam, self.B)).argmax(axis=1)]


def fit_vbs(df, cols, beta=0.9, xi0=0.0, mode="greedy", beam_k=3, noise=1.0, ridge=1.0):
    """Walk the batches in order, choosing s_t per batch via the evidence ratio."""
    batches = sorted(df["batch"].unique())
    sc = StandardScaler().fit(df[df["batch"] == batches[0]][cols])
    d = len(cols) + 1
    beams = [(np.eye(d) * ridge, np.zeros((d, NK)), 0.0, [])]  # (Lambda, B, cum log-ev, flags)

    for b in batches:
        sub = df[df["batch"] == b]
        Xb, Yb = _design(sc.transform(sub[cols])), _onehot(sub["gas_class"].values)
        children = []
        for Lam, B, cum, hist in beams:
            for s in (0, 1):
                Lam_p, B_p = (beta * Lam, beta * B) if s == 1 else (Lam, B)  # s=1 => temper
                ev = bayes_log_evidence(Lam_p, B_p, Xb, Yb, noise)
                LamN, BN = _posterior(Lam_p, B_p, Xb, Yb, noise)
                children.append((LamN, BN, cum + ev + (xi0 if s == 1 else 0.0), hist + [s]))

        if mode == "greedy":
            beams = [max(children[-2:], key=lambda h: h[2])]  # commit immediately
        else:
            children.sort(key=lambda h: -h[2])
            kept, seen = [], set()
            for h in children:
                key = tuple(h[3][-4:])  # diversification: avoid near-identical hypotheses
                if key in seen and kept:
                    continue
                seen.add(key)
                kept.append(h)
                if len(kept) == beam_k:
                    break
            beams = kept

    Lam, B, cum, flags = max(beams, key=lambda h: h[2])
    return VBSModel(sc, Lam, B, flags, cum)


def fit_bayes_static(df, cols, mode, noise=1.0, ridge=1.0):
    """VBS's own baselines: 'once' = batch 1 only, never updated; 'retrain' = all data."""
    batches = sorted(df["batch"].unique())
    sc = StandardScaler().fit(df[df["batch"] == batches[0]][cols])
    src = df[df["batch"] == batches[0]] if mode == "once" else df
    d = len(cols) + 1
    X, Y = _design(sc.transform(src[cols])), _onehot(src["gas_class"].values)
    Lam, B = _posterior(np.eye(d) * ridge, np.zeros((d, NK)), X, Y, noise)
    return VBSModel(sc, Lam, B, [], 0.0)


# VBS needs the batch structure at fit time, so it rides the transductive harness
# (which hands over the training-fold dataframe). It never touches target labels.
def vbs_predict_fn(beta, xi0, mode):
    return lambda tr, X_va, cols: fit_vbs(tr, cols, beta, xi0, mode).predict(X_va)


print("Grid-searching beta / xi_0 (each fold trains only on its own past)...")
vbs_grid = {}
for beta in (0.5, 0.6, 0.7, 0.8, 0.9):
    for p_chg in (0.501, 0.6, 0.7, 0.8, 0.9):
        xi0 = np.log(p_chg / (1 - p_chg))
        s = forward_chaining_cv_transductive(train, feature_cols, vbs_predict_fn(beta, xi0, "greedy"))
        vbs_grid[(beta, p_chg)] = float(np.mean([x[2] for x in s]))

VBS_BETA, VBS_P = max(vbs_grid, key=vbs_grid.get)
VBS_XI0 = np.log(VBS_P / (1 - VBS_P))
print(f"  best: beta={VBS_BETA}, p(change)={VBS_P}  ->  {vbs_grid[(VBS_BETA, VBS_P)]:.4f}\n")

results["VBS greedy (K=1)"] = summarize_cv(
    forward_chaining_cv_transductive(train, feature_cols, vbs_predict_fn(VBS_BETA, VBS_XI0, "greedy")),
    "VBS greedy (K=1)")
results["VBS beam (K=3)"] = summarize_cv(
    forward_chaining_cv_transductive(train, feature_cols, vbs_predict_fn(VBS_BETA, VBS_XI0, "beam")),
    "VBS beam (K=3)")

# VBS's own baselines -- the comparison that actually tests the mechanism
results["Bayes linear: no adaptation (batch 1 only)"] = summarize_cv(
    forward_chaining_cv_transductive(
        train, feature_cols,
        lambda tr, X_va, cols: fit_bayes_static(tr, cols, "once").predict(X_va)),
    "Bayes linear: no adaptation (batch 1 only)")
results["Bayes linear: full retrain every batch"] = summarize_cv(
    forward_chaining_cv_transductive(
        train, feature_cols,
        lambda tr, X_va, cols: fit_bayes_static(tr, cols, "retrain").predict(X_va)),
    "Bayes linear: full retrain every batch")

# sanity: change points should land on real discontinuities, and more beam width
# must never lower the cumulative evidence it is maximising
g = fit_vbs(train, feature_cols, VBS_BETA, VBS_XI0, "greedy")
bm = fit_vbs(train, feature_cols, VBS_BETA, VBS_XI0, "beam")
print(f"\nchange flags per batch 1-9  greedy={g.change_flags}  beam={bm.change_flags}")
print(f"cumulative log-evidence     greedy={g.cum_evidence:.1f}  beam={bm.cum_evidence:.1f}"
      f"  (beam >= greedy: {bm.cum_evidence >= g.cum_evidence - 1e-6})")

Grid-searching beta / xi_0 (each fold trains only on its own past)...


C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler wa

  best: beta=0.9, p(change)=0.9  ->  0.7610



C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler wa

VBS greedy (K=1): macro F1 = 0.7610 +/- 0.2115
  validate on batch 2 (n=1244): 0.2813
  validate on batch 3 (n=1586): 0.7299
  validate on batch 4 (n=161): 0.7258
  validate on batch 5 (n=197): 0.9838
  validate on batch 6 (n=2300): 0.6600
  validate on batch 7 (n=3613): 0.8765
  validate on batch 8 (n=294): 0.8660
  validate on batch 9 (n=470): 0.9647


C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler wa

VBS beam (K=3): macro F1 = 0.7605 +/- 0.2113
  validate on batch 2 (n=1244): 0.2813
  validate on batch 3 (n=1586): 0.7299
  validate on batch 4 (n=161): 0.7258
  validate on batch 5 (n=197): 0.9838
  validate on batch 6 (n=2300): 0.6600
  validate on batch 7 (n=3613): 0.8723
  validate on batch 8 (n=294): 0.8660
  validate on batch 9 (n=470): 0.9647
Bayes linear: no adaptation (batch 1 only): macro F1 = 0.2040 +/- 0.0830
  validate on batch 2 (n=1244): 0.2813
  validate on batch 3 (n=1586): 0.0981
  validate on batch 4 (n=161): 0.1818
  validate on batch 5 (n=197): 0.1680
  validate on batch 6 (n=2300): 0.1311
  validate on batch 7 (n=3613): 0.1360
  validate on batch 8 (n=294): 0.3282
  validate on batch 9 (n=470): 0.3076


C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


Bayes linear: full retrain every batch: macro F1 = 0.7605 +/- 0.2113
  validate on batch 2 (n=1244): 0.2813
  validate on batch 3 (n=1586): 0.7299
  validate on batch 4 (n=161): 0.7258
  validate on batch 5 (n=197): 0.9838
  validate on batch 6 (n=2300): 0.6600
  validate on batch 7 (n=3613): 0.8723
  validate on batch 8 (n=294): 0.8660
  validate on batch 9 (n=470): 0.9647

change flags per batch 1-9  greedy=[0, 0, 0, 0, 1, 0, 1, 1, 1]  beam=[0, 0, 0, 0, 0, 1, 1, 1, 1]
cumulative log-evidence     greedy=-59451.8  beam=-59450.9  (beam >= greedy: True)


## DRCA vs Knowledge Distillation — data-perspective vs model-perspective adaptation

Both are unsupervised on the target (no labels for the batch being predicted).

**DRCA (data perspective)** learns a linear projection into a subspace that minimises *between-domain* scatter (source mean vs target mean) while preserving *within-domain* scatter so the classes don't collapse. Top-`d` eigenvectors of `(S_b + λI)⁻¹(S_wˢ + α·S_wᵀ)`.

> Note `S_b = (m_s − m_t)(m_s − m_t)ᵀ` is **rank 1**, hence singular and not invertible as written — it needs the ridge `λI`. That is the main implementation trap in this method.

**KD (model perspective)** trains a teacher FCNN [100, 50, 20] on the labelled source, uses it to generate *soft* labels for both source and unlabelled target, then trains a student on the union against those soft targets at temperature `T`. Soft labels carry inter-class similarity structure that one-hot labels throw away, which is what lets the student generalise onto the target.

**KD-DRCA hybrid**: project with DRCA first, then distil. The paper reports this underperforms plain KD — worth checking rather than assuming.

In [18]:
# --- 6. DRCA ------------------------------------------------------------------
def drca_projection(Xs, Xt, d, alpha, ridge=1e-3):
    """Top-d eigenvectors of (S_b + ridge*I)^-1 (S_w^S + alpha*S_w^T).
    S_b is rank 1 (outer product of the mean difference) => singular => the ridge
    is required, not optional."""
    diff = (Xs.mean(0) - Xt.mean(0)).reshape(-1, 1)
    A = np.cov(Xs, rowvar=False) + alpha * np.cov(Xt, rowvar=False)
    B = diff @ diff.T + ridge * np.eye(Xs.shape[1])
    w, V = linalg.eigh(A, B)
    return np.real(V[:, np.argsort(-w)[:d]])


def make_drca_predict_fn(d, alpha):
    def predict_fn(tr, X_va, cols):
        sc = StandardScaler().fit(tr[cols])
        Xs, Xt = sc.transform(tr[cols]), sc.transform(X_va)
        P = drca_projection(Xs, Xt, d, alpha)
        return RF().fit(Xs @ P, tr["gas_class"]).predict(Xt @ P)
    return predict_fn


# sanity: the projection must actually shrink the domain gap
_tr, _va = train[train["batch"] < 9], train[train["batch"] == 9]
_sc = StandardScaler().fit(_tr[feature_cols])
_Xs, _Xt = _sc.transform(_tr[feature_cols]), _sc.transform(_va[feature_cols])


def _gap_ratio(Xs, Xt):
    return (np.linalg.norm(Xs.mean(0) - Xt.mean(0))
            / np.sqrt(np.trace(np.cov(Xs, rowvar=False)) + np.trace(np.cov(Xt, rowvar=False))))


_P = drca_projection(_Xs, _Xt, 50, 1e-3)
print(f"DRCA sanity (batch 9): mean-gap / within-scatter  before={_gap_ratio(_Xs, _Xt):.4f}"
      f"  after={_gap_ratio(_Xs @ _P, _Xt @ _P):.4f}")

print("\nDRCA grid (walk-forward):")
drca_grid = {}
for d in (50, 100):
    for alpha in (1e-4, 1e-3, 1e-1, 1, 10, 1000):
        s = forward_chaining_cv_transductive(train, feature_cols, make_drca_predict_fn(d, alpha))
        drca_grid[(d, alpha)] = float(np.mean([x[2] for x in s]))
        print(f"  d={d:<4} alpha={alpha:<7} F1={drca_grid[(d, alpha)]:.4f}")

DRCA_D, DRCA_ALPHA = max(drca_grid, key=drca_grid.get)
print(f"  best: d={DRCA_D}, alpha={DRCA_ALPHA}\n")

results[f"DRCA (d={DRCA_D}, alpha={DRCA_ALPHA})"] = summarize_cv(
    forward_chaining_cv_transductive(train, feature_cols, make_drca_predict_fn(DRCA_D, DRCA_ALPHA)),
    f"DRCA (d={DRCA_D}, alpha={DRCA_ALPHA})")

DRCA sanity (batch 9): mean-gap / within-scatter  before=0.4867  after=0.0001

DRCA grid (walk-forward):


C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler wa

  d=50   alpha=0.0001  F1=0.8166


C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler wa

  d=50   alpha=0.001   F1=0.8201


...

  d=100  alpha=10      F1=0.7122


C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler wa

  d=100  alpha=1000    F1=0.5678
  best: d=50, alpha=0.001



C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler wa

DRCA (d=50, alpha=0.001): macro F1 = 0.8201 +/- 0.1267
  validate on batch 2 (n=1244): 0.6822
  validate on batch 3 (n=1586): 0.9502
  validate on batch 4 (n=161): 0.7288
  validate on batch 5 (n=197): 0.9826
  validate on batch 6 (n=2300): 0.6373
  validate on batch 7 (n=3613): 0.7466
  validate on batch 8 (n=294): 0.9043
  validate on batch 9 (n=470): 0.9291


In [19]:
# --- 7. Knowledge distillation (teacher -> soft labels -> student), on GPU -----
class FCNN(nn.Module):
    """The paper's architecture: 4 layers, hidden [100, 50, 20], ReLU, softmax out."""

    def __init__(self, d_in, n_classes=6):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, 100), nn.ReLU(),
                                 nn.Linear(100, 50), nn.ReLU(),
                                 nn.Linear(50, 20), nn.ReLU(),
                                 nn.Linear(20, n_classes))

    def forward(self, x):
        return self.net(x)


def train_teacher(Xs, ys, epochs=60, bs=1024, lr=1e-3):
    """Source-only => can be cached per fold and reused across evaluation splits."""
    m = FCNN(Xs.shape[1]).to(DEVICE)
    opt = torch.optim.Adam(m.parameters(), lr=lr)
    X = torch.tensor(Xs, dtype=torch.float32, device=DEVICE)
    y = torch.tensor(ys, dtype=torch.long, device=DEVICE)
    for _ in range(epochs):
        m.train()
        for idx in torch.randperm(len(X), device=DEVICE).split(bs):
            loss = F.cross_entropy(m(X[idx]), y[idx])
            opt.zero_grad(); loss.backward(); opt.step()
    m.eval()
    with torch.no_grad():
        acc = (m(X).argmax(1) == y).float().mean().item()
    return m, acc


def train_student(teacher, Xs, Xt, T, epochs=60, bs=1024, lr=1e-3):
    """Student fits the teacher's SOFT labels over source+target inputs.
    Target labels are never used -- only target feature vectors."""
    X = torch.tensor(np.vstack([Xs, Xt]), dtype=torch.float32, device=DEVICE)
    with torch.no_grad():
        soft = F.softmax(teacher(X) / T, dim=1)
    s = FCNN(X.shape[1]).to(DEVICE)
    opt = torch.optim.Adam(s.parameters(), lr=lr)
    for _ in range(epochs):
        s.train()
        for idx in torch.randperm(len(X), device=DEVICE).split(bs):
            loss = F.kl_div(F.log_softmax(s(X[idx]) / T, dim=1), soft[idx],
                            reduction="batchmean") * (T ** 2)   # standard T^2 rescaling
            opt.zero_grad(); loss.backward(); opt.step()
    s.eval()
    return s


def make_kd_predict_fn(T, drca_params=None, diag=None):
    def predict_fn(tr, X_va, cols):
        sc = StandardScaler().fit(tr[cols])
        Xs, Xt = sc.transform(tr[cols]), sc.transform(X_va)
        if drca_params is not None:
            P = drca_projection(Xs, Xt, *drca_params)
            Xs, Xt = Xs @ P, Xt @ P
        teacher, tacc = train_teacher(Xs, tr["gas_class"].values - 1)
        student = train_student(teacher, Xs, Xt, T)
        with torch.no_grad():
            Xs_t = torch.tensor(Xs, dtype=torch.float32, device=DEVICE)
            Xt_t = torch.tensor(Xt, dtype=torch.float32, device=DEVICE)
            agree = (student(Xs_t).argmax(1) == teacher(Xs_t).argmax(1)).float().mean().item()
            preds = student(Xt_t).argmax(1).cpu().numpy() + 1
        if diag is not None:
            diag.append((tacc, agree))
        return preds
    return predict_fn


print("KD temperature grid (walk-forward, GPU):")
kd_grid, kd_diag = {}, []
for T in (1, 2, 5, 25):
    s = forward_chaining_cv_transductive(train, feature_cols, make_kd_predict_fn(T, diag=kd_diag))
    kd_grid[T] = float(np.mean([x[2] for x in s]))
    print(f"  T={T:<4} F1={kd_grid[T]:.4f}")

KD_T = max(kd_grid, key=kd_grid.get)
tacc, agree = np.mean([d[0] for d in kd_diag]), np.mean([d[1] for d in kd_diag])
print(f"  best T={KD_T}")
print(f"  KD sanity: teacher source acc={tacc:.3f} (must be high), "
      f"student-teacher agreement on source={agree:.3f} (distillation worked)\n")

results[f"KD (T={KD_T})"] = summarize_cv(
    forward_chaining_cv_transductive(train, feature_cols, make_kd_predict_fn(KD_T)), f"KD (T={KD_T})")
results["KD-DRCA hybrid"] = summarize_cv(
    forward_chaining_cv_transductive(
        train, feature_cols, make_kd_predict_fn(KD_T, drca_params=(DRCA_D, DRCA_ALPHA, 1e-3))),
    "KD-DRCA hybrid")

KD temperature grid (walk-forward, GPU):


C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler wa

  T=1    F1=0.7437


C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler wa

  T=2    F1=0.7691


...

  T=25   F1=0.7573
  best T=5
  KD sanity: teacher source acc=0.990 (must be high), student-teacher agreement on source=0.983 (distillation worked)



C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler wa

KD (T=5): macro F1 = 0.7547 +/- 0.1307
  validate on batch 2 (n=1244): 0.6139
  validate on batch 3 (n=1586): 0.5979
  validate on batch 4 (n=161): 0.7824
  validate on batch 5 (n=197): 0.9826
  validate on batch 6 (n=2300): 0.6443
  validate on batch 7 (n=3613): 0.7415
  validate on batch 8 (n=294): 0.9200
  validate on batch 9 (n=470): 0.7545


C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler wa

KD-DRCA hybrid: macro F1 = 0.7542 +/- 0.1034
  validate on batch 2 (n=1244): 0.6163
  validate on batch 3 (n=1586): 0.7702
  validate on batch 4 (n=161): 0.6390
  validate on batch 5 (n=197): 0.8264
  validate on batch 6 (n=2300): 0.6491
  validate on batch 7 (n=3613): 0.8792
  validate on batch 8 (n=294): 0.8997
  validate on batch 9 (n=470): 0.7539


## Statistical protocol: 30 splits per batch, four metrics, paired t-tests

Every number above is a single point estimate — one score per fold, no error bar. With 20+ configs compared on 8 folds, the top of that leaderboard is partly noise (OSC k=2 at 0.8467 vs k=3 at 0.8454 is not a real difference).

The protocol below fixes that. For each fold (train = batches `< n`, target = batch `n`):

1. Draw **30 random 50/50 splits** of batch `n` into an *adapt half* and an *eval half*, from a fixed seed so **every method sees identical splits** — that is what makes the t-tests paired.
2. Adaptive methods may use the adapt half's **features** (never labels); source-only methods ignore it entirely and are simply fit once per fold.
3. Score every method on the eval half: **macro-F1, accuracy, macro-precision, macro-recall**.

Two significance tests are reported against the RF baseline:
- **all-splits** (8 × 30 = 240 paired differences) — the protocol from the paper, but splits within a batch share training data and overlap heavily, so this p-value is **optimistic**;
- **fold-level** (8 paired fold-mean differences) — far more conservative and closer to independent. **Claims of significance lean on this one.**

In [20]:
# --- 8. Multi-split protocol harness ------------------------------------------
import time
from scipy import stats
from sklearn.metrics import precision_score, recall_score

N_SPLITS = 30


def make_splits(df, n_splits=N_SPLITS, seed=0, batch_col="batch"):
    """Per target batch: n_splits fixed 50/50 (adapt, eval) positional index pairs.
    Shared by every method, which is what makes the paired tests valid."""
    rng = np.random.RandomState(seed)
    splits = {}
    for vb in sorted(df[batch_col].unique())[1:]:
        idx = np.where(df[batch_col].values == vb)[0]
        splits[vb] = []
        for _ in range(n_splits):
            perm = rng.permutation(idx)
            h = len(perm) // 2
            splits[vb].append((np.sort(perm[:h]), np.sort(perm[h:])))
    return splits


def _metrics(y_true, y_pred):
    return dict(f1=f1_score(y_true, y_pred, average="macro"),
                acc=accuracy_score(y_true, y_pred),
                prec=precision_score(y_true, y_pred, average="macro", zero_division=0),
                rec=recall_score(y_true, y_pred, average="macro", zero_division=0))


def evaluate_method(name, spec, splits, rows, verbose=True):
    """spec = dict(df, cols, kind, fn)
      kind='source_only': fn(tr_df, cols) -> model with .predict(X)   [fit ONCE per fold]
      kind='adaptive'   : fn(tr_df, X_adapt, X_eval, cols) -> preds   [refit per split]"""
    df, cols, kind, fn = spec["df"], spec["cols"], spec["kind"], spec["fn"]
    X_all, y_all = df[cols].values, df["gas_class"].values
    t0 = time.time()
    for vb, per in splits.items():
        tr = df[df["batch"] < vb]
        model = fn(tr, cols) if kind == "source_only" else None
        for si, (a_idx, e_idx) in enumerate(per):
            preds = (model.predict(X_all[e_idx]) if kind == "source_only"
                     else fn(tr, X_all[a_idx], X_all[e_idx], cols))
            rows.append(dict(method=name, batch=vb, split=si, **_metrics(y_all[e_idx], preds)))
    if verbose:
        print(f"  {name:<46} [{time.time() - t0:6.1f}s]")


splits = make_splits(train)
# protocol sanity: halves must be disjoint, drawn from the right batch, and identical for all methods
for vb, per in splits.items():
    for a, e in per:
        assert len(np.intersect1d(a, e)) == 0, "adapt/eval overlap"
        assert set(train["batch"].values[np.concatenate([a, e])]) == {vb}
print(f"splits OK: {N_SPLITS} per batch x {len(splits)} target batches, no adapt/eval overlap\n")


# --- wrappers so every method exposes the same two interfaces -----------------
class _Wrap:
    def __init__(self, m, pre=None): self.m, self.pre = m, pre
    def predict(self, X): return self.m.predict(self.pre(X) if self.pre else X)


class _OffsetWrap:      # XGBoost predicts 0-indexed labels
    def __init__(self, m): self.m = m
    def predict(self, X): return self.m.predict(X) + 1


def fit_plain(factory, sw_fn=None, offset=0):
    def fit(tr, cols):
        m = factory()
        kw = {"sample_weight": sw_fn(tr)} if sw_fn else {}
        m.fit(tr[cols], tr["gas_class"] - offset, **kw)
        return _OffsetWrap(m) if offset else _Wrap(m)
    return fit


def fit_osc(k):
    def fit(tr, cols):
        sc = StandardScaler().fit(tr[cols])
        Xtr = sc.transform(tr[cols])
        P = osc_projector(tr, Xtr, k)
        clf = RF().fit(Xtr @ P, tr["gas_class"])
        return _Wrap(clf, pre=lambda X: sc.transform(X) @ P)
    return fit


def fit_svm_ens(pre_aging):
    def fit(tr, cols):
        mm = MinMaxScaler(feature_range=(-1, 1))
        trs = tr.copy()
        trs[cols] = mm.fit_transform(tr[cols])
        ens = DriftAdaptiveSVMEnsemble(classes=classes, pre_aging=pre_aging).fit(trs, cols)
        return _Wrap(ens, pre=lambda X: mm.transform(X))
    return fit


class _NetWrap:
    def __init__(self, model, sc): self.model, self.sc = model, sc
    def predict(self, X):
        with torch.no_grad():
            t = torch.tensor(self.sc.transform(X), dtype=torch.float32, device=DEVICE)
            return self.model(t)[0].argmax(1).cpu().numpy() + 1


def fit_mlp_control(tr, cols):
    """The DANN architecture with the adversary switched off -- source-only, so it is
    fit once per fold like any other non-adaptive model."""
    sc = StandardScaler().fit(tr[cols])
    Xs = sc.transform(tr[cols])
    model = DANN(Xs.shape[1]).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    X = torch.tensor(Xs, dtype=torch.float32, device=DEVICE)
    y = torch.tensor(tr["gas_class"].values - 1, dtype=torch.long, device=DEVICE)
    for _ in range(40):
        model.train()
        for idx in torch.randperm(len(X), device=DEVICE).split(1024):
            loss = F.cross_entropy(model(X[idx], 0.0)[0], y[idx])
            opt.zero_grad(); loss.backward(); opt.step()
    model.eval()
    return _NetWrap(model, sc)


def dann_adaptive(tr, X_adapt, X_eval, cols):
    sc = StandardScaler().fit(tr[cols])
    preds, _, _ = fit_dann(sc.transform(tr[cols]), tr["gas_class"].values - 1,
                           sc.transform(np.vstack([X_adapt, X_eval])), adversarial=True)
    return preds[len(X_adapt):] + 1          # adapt on both halves' features, score the eval half


def coral_adaptive(kind):
    def fn(tr, X_adapt, X_eval, cols):
        if kind == "naive":
            sc = StandardScaler().fit(tr[cols])
            Xs, Xa, Xe = sc.transform(tr[cols]), sc.transform(X_adapt), sc.transform(X_eval)
        else:
            Xs = StandardScaler().fit_transform(tr[cols])
            sct = StandardScaler().fit(X_adapt)
            Xa, Xe = sct.transform(X_adapt), sct.transform(X_eval)
        if kind == "std_only":
            return RF().fit(Xs, tr["gas_class"]).predict(Xe)
        d = Xs.shape[1]
        eps = 1.0 if kind == "naive" else 1e-3
        A = np.real(linalg.sqrtm(linalg.inv(np.cov(Xs, rowvar=False) + eps * np.eye(d)))
                    @ linalg.sqrtm(np.cov(Xa, rowvar=False) + eps * np.eye(d)))
        return RF().fit(Xs @ A, tr["gas_class"]).predict(Xe)
    return fn


def drca_adaptive(tr, X_adapt, X_eval, cols):
    sc = StandardScaler().fit(tr[cols])
    Xs, Xa, Xe = sc.transform(tr[cols]), sc.transform(X_adapt), sc.transform(X_eval)
    P = drca_projection(Xs, Xa, DRCA_D, DRCA_ALPHA)
    return RF().fit(Xs @ P, tr["gas_class"]).predict(Xe @ P)


_teacher_cache = {}


def kd_adaptive(use_drca):
    def fn(tr, X_adapt, X_eval, cols):
        sc = StandardScaler().fit(tr[cols])
        Xs, Xa, Xe = sc.transform(tr[cols]), sc.transform(X_adapt), sc.transform(X_eval)
        if use_drca:
            P = drca_projection(Xs, Xa, DRCA_D, DRCA_ALPHA)
            Xs, Xa, Xe = Xs @ P, Xa @ P, Xe @ P
            teacher, _ = train_teacher(Xs, tr["gas_class"].values - 1)   # depends on P, can't cache
        else:
            key = (len(tr), use_drca)          # teacher is source-only -> cache per fold
            if key not in _teacher_cache:
                _teacher_cache[key] = train_teacher(Xs, tr["gas_class"].values - 1)[0]
            teacher = _teacher_cache[key]
        student = train_student(teacher, Xs, Xa, KD_T)
        with torch.no_grad():
            t = torch.tensor(Xe, dtype=torch.float32, device=DEVICE)
            return student(t).argmax(1).cpu().numpy() + 1
    return fn


METHOD_SPECS = {
    "RF (baseline: raw, unweighted)": dict(df=train, cols=feature_cols, kind="source_only",
                                           fn=fit_plain(RF)),
    "RF (scaled + balanced)": dict(df=train_scaled, cols=feature_cols_v2, kind="source_only",
                                   fn=fit_plain(lambda: RandomForestClassifier(
                                       n_estimators=300, random_state=42, n_jobs=-1, class_weight="balanced"))),
    "LightGBM (scaled + balanced)": dict(df=train_scaled, cols=feature_cols_v2, kind="source_only",
                                         fn=fit_plain(lambda: LGBMClassifier(
                                             n_estimators=300, random_state=42, class_weight="balanced", verbosity=-1))),
    "XGBoost (scaled + balanced)": dict(df=train_scaled, cols=feature_cols_v2, kind="source_only",
                                        fn=fit_plain(lambda: XGBClassifier(
                                            n_estimators=300, random_state=42, eval_metric="mlogloss"),
                                            sw_fn=lambda df: compute_sample_weight("balanced", df["gas_class"]),
                                            offset=1)),
    "Baseline correction (per-sensor ratios)": dict(df=train_bc, cols=feature_cols_bc,
                                                    kind="source_only", fn=fit_plain(RF)),
    "Dynamic RF ensemble (per-batch, recency x acc)": dict(
        df=train, cols=feature_cols, kind="source_only",
        fn=lambda tr, cols: _Wrap(DynamicRFEnsemble(classes).fit(tr, cols))),
    "Recency-weighted RF (half-life=2.0 batches)": dict(
        df=train, cols=feature_cols, kind="source_only",
        fn=fit_plain(RF, sw_fn=make_recency_weights(2.0))),
    "SVM 2D ensemble (pre-aging)": dict(df=train, cols=feature_cols, kind="source_only",
                                        fn=fit_svm_ens(True)),
    "SVM 2D ensemble (no pre-aging)": dict(df=train, cols=feature_cols, kind="source_only",
                                           fn=fit_svm_ens(False)),
    "MLP control (no adaptation)": dict(df=train, cols=feature_cols, kind="source_only",
                                        fn=fit_mlp_control),
    "VBS greedy (K=1)": dict(df=train, cols=feature_cols, kind="source_only",
                             fn=lambda tr, cols: fit_vbs(tr, cols, VBS_BETA, VBS_XI0, "greedy")),
    "VBS beam (K=3)": dict(df=train, cols=feature_cols, kind="source_only",
                           fn=lambda tr, cols: fit_vbs(tr, cols, VBS_BETA, VBS_XI0, "beam")),
    "Bayes linear: no adaptation (batch 1 only)": dict(
        df=train, cols=feature_cols, kind="source_only",
        fn=lambda tr, cols: fit_bayes_static(tr, cols, "once")),
    "Bayes linear: full retrain every batch": dict(
        df=train, cols=feature_cols, kind="source_only",
        fn=lambda tr, cols: fit_bayes_static(tr, cols, "retrain")),
    **{f"OSC / component correction (k={k})": dict(df=train, cols=feature_cols,
                                                   kind="source_only", fn=fit_osc(k))
       for k in (1, 2, 3)},
    "CORAL (naive: shared source scaler)": dict(df=train, cols=feature_cols, kind="adaptive",
                                                fn=coral_adaptive("naive")),
    "CORAL (per-domain centering)": dict(df=train, cols=feature_cols, kind="adaptive",
                                         fn=coral_adaptive("per_domain")),
    "Ablation: per-domain standardization only": dict(df=train, cols=feature_cols, kind="adaptive",
                                                      fn=coral_adaptive("std_only")),
    "DANN (domain-adversarial)": dict(df=train, cols=feature_cols, kind="adaptive", fn=dann_adaptive),
    f"DRCA (d={DRCA_D}, alpha={DRCA_ALPHA})": dict(df=train, cols=feature_cols, kind="adaptive",
                                                   fn=drca_adaptive),
    f"KD (T={KD_T})": dict(df=train, cols=feature_cols, kind="adaptive", fn=kd_adaptive(False)),
    "KD-DRCA hybrid": dict(df=train, cols=feature_cols, kind="adaptive", fn=kd_adaptive(True)),
}
print(f"{len(METHOD_SPECS)} methods registered for the multi-split protocol.")

splits OK: 30 per batch x 8 target batches, no adapt/eval overlap

24 methods registered for the multi-split protocol.


In [21]:
# --- 9. Run the protocol and rank everything ----------------------------------
print(f"Running {len(METHOD_SPECS)} methods x {len(splits)} folds x {N_SPLITS} splits...\n")
proto_rows = []
for _name, _spec in METHOD_SPECS.items():
    evaluate_method(_name, _spec, splits, proto_rows)

proto = pd.DataFrame(proto_rows)
BASELINE = "RF (baseline: raw, unweighted)"

agg = proto.groupby("method")[["f1", "acc", "prec", "rec"]].mean()
sd = proto.groupby("method")["f1"].std()
base_rows = proto[proto.method == BASELINE].sort_values(["batch", "split"])
base_fold = base_rows.groupby("batch").f1.mean()

table = []
for m in agg.index:
    cur = proto[proto.method == m].sort_values(["batch", "split"])
    if m == BASELINE:
        p_all = p_fold = np.nan
    else:
        p_all = stats.ttest_rel(cur.f1.values, base_rows.f1.values).pvalue
        p_fold = stats.ttest_rel(cur.groupby("batch").f1.mean().values, base_fold.values).pvalue
    table.append(dict(method=m, f1=agg.loc[m, "f1"], sd=sd[m], acc=agg.loc[m, "acc"],
                      prec=agg.loc[m, "prec"], rec=agg.loc[m, "rec"],
                      delta=agg.loc[m, "f1"] - agg.loc[BASELINE, "f1"],
                      p_all=p_all, p_fold=p_fold))
table = pd.DataFrame(table).sort_values("f1", ascending=False).reset_index(drop=True)

print(f"\n{'='*118}")
print(f"FINAL RANKING  --  {len(splits)} batches x {N_SPLITS} splits, paired vs '{BASELINE}'")
print(f"{'='*118}")
print(f"{'method':<46}{'macroF1':>9}{'sd':>7}{'acc':>7}{'prec':>7}{'rec':>7}"
      f"{'delta':>8}{'p(all)':>10}{'p(fold)':>9}  sig")
print("-" * 118)
for _, r in table.iterrows():
    sig = "" if np.isnan(r.p_fold) else ("**" if r.p_fold < 0.01 else "*" if r.p_fold < 0.05 else "ns")
    if not np.isnan(r.p_fold) and r.delta < 0 and r.p_fold < 0.05:
        sig += " (worse)"
    print(f"{r.method:<46}{r.f1:>9.4f}{r.sd:>7.3f}{r.acc:>7.4f}{r.prec:>7.4f}{r.rec:>7.4f}"
          f"{r.delta:>+8.4f}{r.p_all:>10.2e}{r.p_fold:>9.4f}  {sig}")
print("-" * 118)
print("sig uses the CONSERVATIVE fold-level test (n=8).  ** p<0.01, * p<0.05, ns = not significant.")
print("p(all) over 240 paired splits is optimistic: splits within a batch share training data.")

# Does the previous winner's margin actually survive?
_o2 = proto[proto.method == "OSC / component correction (k=2)"].sort_values(["batch", "split"])
_o3 = proto[proto.method == "OSC / component correction (k=3)"].sort_values(["batch", "split"])
_p23 = stats.ttest_rel(_o2.groupby("batch").f1.mean().values, _o3.groupby("batch").f1.mean().values).pvalue
print(f"\nOSC k=2 vs k=3 head-to-head: fold-level p={_p23:.3f} -> "
      f"{'indistinguishable' if _p23 > 0.05 else 'a real difference'}")

Running 24 methods x 8 folds x 30 splits...



C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feat

  RF (baseline: raw, unweighted)                 [  13.3s]


C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feat

  RF (scaled + balanced)                         [  14.3s]


...

  DRCA (d=50, alpha=0.001)                       [ 118.6s]


C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler wa

  KD (T=5)                                       [ 133.5s]


C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler wa

  KD-DRCA hybrid                                 [ 216.0s]

FINAL RANKING  --  8 batches x 30 splits, paired vs 'RF (baseline: raw, unweighted)'
method                                          macroF1     sd    acc   prec    rec   delta    p(all)  p(fold)  sig
----------------------------------------------------------------------------------------------------------------------
OSC / component correction (k=2)                 0.8477  0.124 0.8667 0.8562 0.8798 +0.0671  8.03e-41   0.0217  *
OSC / component correction (k=3)                 0.8456  0.133 0.8729 0.8647 0.8749 +0.0649  7.66e-33   0.0456  *
OSC / component correction (k=1)                 0.8299  0.121 0.8570 0.8409 0.8730 +0.0492  1.78e-51   0.0055  **
Baseline correction (per-sensor ratios)          0.8266  0.144 0.8616 0.8484 0.8586 +0.0460  9.85e-21   0.1067  ns
DRCA (d=50, alpha=0.001)                         0.8102  0.154 0.8438 0.8359 0.8452 +0.0295  4.17e-08   0.2851  ns
KD-DRCA hybrid                                 

In [22]:
# --- 10. Task 1: train on batch 1 only, score each later batch ----------------
# Task 2 (above) keeps enlarging the training set, which masks how badly raw drift
# hurts. Task 1 freezes the model at batch 1 so the degradation with time gap is visible.
task1_methods = {
    "RF (baseline)": fit_plain(RF),
    "Baseline correction": None,          # needs the bc frame, handled below
    "OSC (k=2)": fit_osc(2),
    "SVM 2D ensemble (pre-aging)": fit_svm_ens(True),
}

b1 = train[train["batch"] == 1]
b1_bc = train_bc[train_bc["batch"] == 1]
rows_t1 = []
for name, fit in task1_methods.items():
    if name == "Baseline correction":
        model, src_cols, src_df = fit_plain(RF)(b1_bc, feature_cols_bc), feature_cols_bc, train_bc
    else:
        model, src_cols, src_df = fit(b1, feature_cols), feature_cols, train
    for b in range(2, 10):
        sub = src_df[src_df["batch"] == b]
        rows_t1.append(dict(method=name, batch=b,
                            f1=f1_score(sub["gas_class"], model.predict(sub[src_cols].values),
                                        average="macro")))

t1 = pd.DataFrame(rows_t1).pivot(index="method", columns="batch", values="f1")
print("=== Task 1: trained on batch 1 ONLY, macro-F1 on each later batch ===")
print(t1.round(3).to_string())
print("\nmean over batches 2-9:")
print(t1.mean(axis=1).sort_values(ascending=False).round(4).to_string())
print("\n(Compare with Task 2 above, where the training set grows each step -- the drop from"
      "\n Task 2 to Task 1 is the cost of never recalibrating.)")

C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feat

=== Task 1: trained on batch 1 ONLY, macro-F1 on each later batch ===
batch                            2      3      4      5      6      7      8      9
method                                                                             
Baseline correction          0.595  0.645  0.290  0.274  0.205  0.221  0.223  0.193
OSC (k=2)                    0.678  0.532  0.488  0.458  0.388  0.322  0.199  0.354
RF (baseline)                0.678  0.532  0.488  0.458  0.388  0.322  0.199  0.354
SVM 2D ensemble (pre-aging)  0.522  0.292  0.240  0.254  0.222  0.226  0.304  0.354

mean over batches 2-9:
method
OSC (k=2)                      0.4274
RF (baseline)                  0.4274
Baseline correction            0.3308
SVM 2D ensemble (pre-aging)    0.3018

(Compare with Task 2 above, where the training set grows each step -- the drop from
 Task 2 to Task 1 is the cost of never recalibrating.)


C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


In [23]:
# --- 11. Re-select the winner from the multi-split protocol -------------------
# Prefer the protocol's mean macro-F1 over the old single-point estimates. Where two
# methods are statistically indistinguishable, keep the SIMPLER one rather than chasing
# a difference the data cannot support.
protocol_scores = table.set_index("method")["f1"].to_dict()

# register the new methods for final refitting on all 9 batches -> batch 10
final_fitters[f"DRCA (d={DRCA_D}, alpha={DRCA_ALPHA})"] = (
    lambda tr, te: drca_adaptive(tr, te[feature_cols].values, te[feature_cols].values, feature_cols))
final_fitters[f"KD (T={KD_T})"] = (
    lambda tr, te: kd_adaptive(False)(tr, te[feature_cols].values, te[feature_cols].values, feature_cols))
final_fitters["KD-DRCA hybrid"] = (
    lambda tr, te: kd_adaptive(True)(tr, te[feature_cols].values, te[feature_cols].values, feature_cols))
final_fitters["VBS greedy (K=1)"] = (
    lambda tr, te: fit_vbs(tr, feature_cols, VBS_BETA, VBS_XI0, "greedy").predict(te[feature_cols].values))
final_fitters["VBS beam (K=3)"] = (
    lambda tr, te: fit_vbs(tr, feature_cols, VBS_BETA, VBS_XI0, "beam").predict(te[feature_cols].values))
final_fitters["Bayes linear: no adaptation (batch 1 only)"] = (
    lambda tr, te: fit_bayes_static(tr, feature_cols, "once").predict(te[feature_cols].values))
final_fitters["Bayes linear: full retrain every batch"] = (
    lambda tr, te: fit_bayes_static(tr, feature_cols, "retrain").predict(te[feature_cols].values))
final_fitters["MLP control (no adaptation)"] = (
    lambda tr, te: fit_mlp_control(tr, feature_cols).predict(te[feature_cols].values))
final_fitters["DANN (domain-adversarial)"] = (
    lambda tr, te: dann_adaptive(tr, te[feature_cols].values[:1], te[feature_cols].values, feature_cols))

best_name = table.iloc[0]["method"]
best_f1 = table.iloc[0]["f1"]

# if the runner-up is statistically tied AND simpler, say so explicitly
runner = table.iloc[1]
tie_p = stats.ttest_rel(
    proto[proto.method == best_name].sort_values(["batch", "split"]).groupby("batch").f1.mean().values,
    proto[proto.method == runner["method"]].sort_values(["batch", "split"]).groupby("batch").f1.mean().values
).pvalue

print(f"Protocol winner : {best_name}  (macro-F1 {best_f1:.4f})")
print(f"Runner-up       : {runner['method']}  ({runner['f1']:.4f}), head-to-head fold-level p={tie_p:.3f}"
      f"  -> {'statistically tied' if tie_p > 0.05 else 'a real gap'}")
print(f"\nSubmission will be written with: {best_name}")
assert best_name in configs or best_name in final_fitters or "SVM 2D" in best_name, \
    f"no refit recipe registered for {best_name}"

Protocol winner : OSC / component correction (k=2)  (macro-F1 0.8477)
Runner-up       : OSC / component correction (k=3)  (0.8456), head-to-head fold-level p=0.854  -> statistically tied

Submission will be written with: OSC / component correction (k=2)


## Final model & submission

Refit the overall winning config (by forward-chaining mean macro F1, across all 6 configs including both SVM ensemble variants) on all 9 labelled batches and predict the hidden batch-10 test set, overwriting `data/submission.csv`.

In [24]:
if best_name in configs:
    best_train_df, best_test_df, best_cols, best_factory, best_sw_fn, best_label_offset = configs[best_name]

    final_model = best_factory()
    y_fit = best_train_df["gas_class"] - best_label_offset
    fit_kwargs = {"sample_weight": best_sw_fn(best_train_df)} if best_sw_fn else {}
    final_model.fit(best_train_df[best_cols], y_fit, **fit_kwargs)

    test_pred_final = final_model.predict(best_test_df[best_cols]) + best_label_offset
elif best_name in final_fitters:
    test_pred_final = final_fitters[best_name](train, test)
else:  # the two SVM 2D ensemble variants
    scaler = MinMaxScaler(feature_range=(-1, 1))
    train_svm_scaled = train.copy()
    train_svm_scaled[feature_cols] = scaler.fit_transform(train[feature_cols])
    X_test_scaled = scaler.transform(test[feature_cols])

    final_model = DriftAdaptiveSVMEnsemble(classes=classes, pre_aging=best_name == "SVM 2D ensemble (pre-aging)")
    final_model.fit(train_svm_scaled, feature_cols)
    test_pred_final = final_model.predict(X_test_scaled)

submission_final = pd.DataFrame({
    "measurement_id": test["measurement_id"],
    "gas_class": test_pred_final,
})
assert list(submission_final.columns) == list(sample_sub.columns)
assert len(submission_final) == len(sample_sub)
assert (submission_final["measurement_id"].values == sample_sub["measurement_id"].values).all()
assert submission_final["gas_class"].isin(range(1, 7)).all()
assert submission_final["gas_class"].notna().all()

submission_final.to_csv("data/submission.csv", index=False)
print(f"Wrote data/submission.csv using: {best_name}")
print(f"  protocol macro-F1 {protocol_scores[best_name]:.4f} "
      f"({len(splits)} batches x {N_SPLITS} splits)")
print("\nPredicted class distribution vs the 600/class the competition states for the test set:")
_vc = submission_final["gas_class"].value_counts().sort_index()
for c in range(1, 7):
    print(f"  class {c}: {_vc.get(c, 0):>5}   ({_vc.get(c, 0) - 600:+d} vs uniform)")
print(f"  total absolute deviation from uniform: {int((_vc - 600).abs().sum())}")

Wrote data/submission.csv using: OSC / component correction (k=2)
  protocol macro-F1 0.8477 (8 batches x 30 splits)

Predicted class distribution vs the 600/class the competition states for the test set:
  class 1:   947   (+347 vs uniform)
  class 2:   485   (-115 vs uniform)
  class 3:   450   (-150 vs uniform)
  class 4:   349   (-251 vs uniform)
  class 5:   468   (-132 vs uniform)
  class 6:   901   (+301 vs uniform)
  total absolute deviation from uniform: 1296
